# ⚽ Análisis de Estadísticas de Partidos de Fútbol con Pandas

Este notebook utiliza Pandas para el análisis de datos de fútbol.

In [ ]:
# Install dependencies
!pip install pandas matplotlib

In [ ]:
# Importar dependencias

# Pandas
import pandas as pd

# Importar Matplotlib
import matplotlib.pyplot as plt

---

## 🥅 Goleadores por Partido (`goalscorers.csv`)

### Carga inicial

In [ ]:
scorers_df = pd.read_csv("goalscorers.csv")

### Exploración inicial

In [ ]:
scorers_df.head().to_csv("scorers_head.csv", index=False)

### Goles por equipo
Cálculo de goles por equipo (excluyendo goles en propia puerta)

In [ ]:
# Agrupamos por equipo y contamos cuántos goles han marcado,
# filtrando previamente los goles en propia puerta para no alterar el dato real de goles a favor
goals_by_team = (
    scorers_df
        .loc[scorers_df["own_goal"].isna()]
        .groupby("team")
        .size()
        .reset_index(name="goals")
        .sort_values("goals", ascending=False)
)

# Guardar en un fichero los 5 primeros
goals_by_team.head(5).to_csv("goals_by_team_top5.csv", index=False)

### Top goleadores
Cálculo del top de goleadores (excluyendo goles en propia puerta)

In [ ]:
# Agrupamos por jugador y contamos cuántos goles han marcado,
# filtrando previamente los goles en propia puerta para no alterar el dato real de goles a favor
top_scorers = (
    scorers_df
        .loc[scorers_df["own_goal"].isna()]
        .groupby("scorer")
        .size()
        .reset_index(name="goals")
        .sort_values("goals", ascending=False)
)

# Guardar en un fichero los 5 primeros
top_scorers.head(5).to_csv("top_scorers_top5.csv", index=False)

### Distribución de goles por tramos de partido
Cálculo de goles por los tramos en minutos de partido (excluyendo goles en propia puerta)

In [ ]:
# Crear tramos de minutos
max_minute = int(scorers_df["minute"].max())

n_bins = max_minute // 15 
bins = [i * 15 for i in range(n_bins + 1)]
bins.append(max_minute)

labels = [
    f"{bins[i] + 1}+"
    if i == len(bins) - 2
    else f"{bins[i] + 1}-{bins[i + 1]}"
    for i in range(len(bins) - 1)
]

scorers_df["minute_bin"] = pd.cut(
    scorers_df["minute"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

# Contar goles por tramo (excluyendo propios en este dataset own_goal es NaN cuando NO lo es)
goals_by_bin = (
    scorers_df
    .loc[scorers_df["own_goal"].isna()]
    .groupby("minute_bin", observed=True)
    .size()
)

# Pintar gráfico de barras
plt.figure(figsize=(8, 5))
goals_by_bin.plot(kind="bar")

plt.xlabel("Tramo de minuto")
plt.ylabel("Número de goles")
plt.title("Distribución de goles por tramos de partido")
plt.xticks(rotation=0)
plt.tight_layout()

# Guardar el gráfico como PNG
plt.savefig("goals_by_minute.png", dpi=300, bbox_inches="tight")